# 06: Niche/domain annotation of reconstructed cells

Read reconstructed molecular states, apply the trusted niche/domain annotation hierarchy, and save annotated volumes.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import anndata as ad
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/03_3d").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "00029_native_v2")
OUT = ROOT / "outputs/03_3d" / RUN_ID
MODE = "path"
SPATIAL_KEY = "spatial_st_corrected"
Z_KEY = "z_um"
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("SPAGC_RUN_ID must be a safe directory name")
ARTIFACTS = ROOT / "outputs/03_3d" / RUN_ID
VOLUME = ARTIFACTS / "reconstructed_volume.h5ad"
if MODE not in ("baseline", "uot", "path"):
    raise ValueError("Unsupported run mode or model mode.")
print({"model_mode": MODE, "output": str(OUT)})
TRANSFER_MODELS = ROOT / "models/label_transfer"
GROUP_KEY = None
SLAB_THICKNESS_UM = 10.0
CONFIDENCE_THRESHOLD = 0.5
RUN_COMPACT_EXPORT = True
USE_NUCLEUS_PATH = False
if USE_NUCLEUS_PATH and MODE != "path":
    raise ValueError(
        "The optional supplied-nucleus experiment requires full mode and morphology paths."
    )

In [ ]:
from spagc._io import require_new
from spagc.reconstruction.annotation import predict_volume, write_compact

Select the real reconstructed volume and trusted reference PCA/MLP bundle. Required model genes are aligned explicitly.

In [ ]:
(input_path, model_dir) = (VOLUME, TRANSFER_MODELS)
if not input_path.is_file():
    raise FileNotFoundError(f"Reconstruct the volume first: {input_path}")

## 2. Transfer niche and domain annotations

Stream reconstructed expression through the saved PCA, aggregate within independent physical-depth slabs, predict niches, and infer domains from predicted niche organization. Save full and sidecar predictions and inspect confidence summaries.

Project reconstructed expression through the trusted annotation models and infer niches and domains on independent two-dimensional depth slabs.

In [ ]:
sidecar_path = require_new(OUT / "niche_domain_predictions.h5ad")
annotated_path = require_new(OUT / "annotated_volume.h5ad")
summary_path = require_new(OUT / "prediction_summary.csv")
predict_volume(
    input_path=input_path,
    model_dir=model_dir,
    sidecar_output=sidecar_path,
    full_output=annotated_path,
    summary_output=summary_path,
    spatial_key=SPATIAL_KEY,
    z_key=Z_KEY,
    group_key=GROUP_KEY,
    slab_thickness=SLAB_THICKNESS_UM,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    delaunay_percentile=99.0,
    read_batch_size=50000,
    max_slabs=None,
)
result = ad.read_h5ad(sidecar_path)
assert result.n_obs > 0 and (not result.obs.pred_domain.isna().any())
assert result.obs.pred_niche_conf.between(0, 1).all()
display(pd.read_csv(summary_path))

## 3. Save a compact annotated volume

Optionally copy only the declared hard annotations after checking exact cell identifiers and expression shape against the base volume.

Copy selected hard annotations only after checking exact cell order and expression shape.

In [ ]:
if RUN_COMPACT_EXPORT:
    columns = [
        "pred_niche",
        "pred_niche_conf",
        "pred_domain",
        "pred_domain_conf",
        "gcsa_z_slab_um",
    ]
    compact = require_new(OUT / "compact_annotated_volume.h5ad")
    write_compact(
        base_path=input_path,
        annotations_path=annotated_path,
        output_path=compact,
        columns=columns,
        comparison_chunk_size=100000,
    )